# SQL Master Decision Tree

**Purpose:** One unified flowchart to navigate from any SQL problem statement to the right approach. Instead of choosing between three separate strategy guides, start here — the tree will route you to the correct technique and tell you which guide to dig into for details.

**How to use this guide:** Read the problem statement, then walk down the tree answering each question. You'll land on the right pattern in under 30 seconds.

---

### Related Guides

| Guide | Use When |
|---|---|
| <a href='sql_single_table_query_strategies.html'>Single-Table Strategies</a> | One table, need to filter / group / rank / compare rows |
| <a href='sql_multi_table_query_strategies.html'>Multi-Table Strategies</a> | Multiple tables, need to choose the right join |
| <a href='sql_combined_strategy_patterns.html'>Combined Strategy Patterns</a> | Join + transformation in the same query |
| <a href='sql_subqueries_select.html'>Subqueries in SELECT</a> | Scalar subquery as column value or denominator |
| <a href='sql_subqueries_where.html'>Subqueries in WHERE</a> | Filter rows by computed threshold |
| <a href='sql_subqueries_from.html'>Subqueries in FROM</a> | Pre-aggregate then query the result |

<hr style="border: 3px solid black;">

<a id='master-tree'></a>

## Step 1: How Many Tables?

This is always the first question. Everything else follows from it.

```
┌─────────────────────────────────────────────────────────┐
│              READ THE PROBLEM STATEMENT                 │
│                                                         │
│  1. How many tables are mentioned?                      │
│  2. What does the expected output look like?            │
│  3. Are there aggregation words (total, average, %)?    │
└────────────────────────┬────────────────────────────────┘
                         │
                         ▼
          ┌──────────────────────────────┐
          │  How many tables are         │
          │  involved in this problem?   │
          └──────┬───────────────┬───────┘
                 │               │
                 ▼               ▼
          ┌────────────┐  ┌──────────────────┐
          │  ONE TABLE │  │  TWO OR MORE     │
          │            │  │  TABLES          │
          │  Go to     │  │                  │
          │  Step 2A   │  │  Go to Step 2B   │
          └────────────┘  └──────────────────┘
```

<hr style="border: 3px solid black;">

<a id='single-table-path'></a>

## Step 2A: Single-Table Path

You have one table. Walk down and stop at the first YES.

```
┌───────────────────────────────────────┐
│   SINGLE TABLE — What am I doing?     │
└──────────────────┬────────────────────┘
                   │
                   ▼
┌──────────────────────────────┐  YES   ┌──────────────────────────────┐
│  Just filtering rows by      ├───────►│  WHERE + ORDER BY            │
│  conditions on columns?      │        │                              │
└──────────┬───────────────────┘        └──────────────────────────────┘
           │ NO
           ▼
┌──────────────────────────────┐  YES   ┌──────────────────────────────┐
│  Does it involve dates?      ├───────►│  See DATE FORK (Step 3)      │
└──────────┬───────────────────┘        └──────────────────────────────┘
           │ NO
           ▼
┌──────────────────────────────┐  YES   ┌──────────────────────────────┐
│  Am I comparing rows?        ├───────►│  LAG() / LEAD()              │
│  (prev vs next, row pairs)   │        │                              │
└──────────┬───────────────────┘        └──────────────────────────────┘
           │ NO
           ▼
┌──────────────────────────────┐  YES   ┌──────────────────────────────┐
│  Am I collapsing rows into   ├───────►│  GROUP BY + aggregate func   │
│  summary? (count, sum, avg)  │        │                              │
└──────────┬───────────────────┘        └──────────────────────────────┘
           │ NO
           ▼
┌──────────────────────────────┐  YES   ┌──────────────────────────────┐
│  Filtering AFTER grouping?   ├───────►│  GROUP BY + HAVING           │
│  ("groups where count > 3")  │        │                              │
└──────────┬───────────────────┘        └──────────────────────────────┘
           │ NO
           ▼
┌──────────────────────────────┐  YES   ┌──────────────────────────────┐
│  Am I ranking rows?          ├───────►│  ROW_NUMBER() / RANK()       │
│  ("top N per group")         │        │                              │
└──────────┬───────────────────┘        └──────────────────────────────┘
           │ NO
           ▼
┌──────────────────────────────┐  YES   ┌──────────────────────────────┐
│  Running total / cumulative? ├───────►│  SUM() OVER(ORDER BY ...)    │
│                              │        │                              │
└──────────┬───────────────────┘        └──────────────────────────────┘
           │ NO
           ▼
┌──────────────────────────────┐  YES   ┌──────────────────────────────┐
│  Labeling / categorizing?    ├───────►│  CASE WHEN                   │
│  ("if X then label Y")       │        │                              │
└──────────┬───────────────────┘        └──────────────────────────────┘
           │ NO
           ▼
┌──────────────────────────────┐  YES   ┌──────────────────────────────┐
│  Pivoting rows → columns?    ├───────►│  CASE + GROUP BY             │
│                              │        │                              │
└──────────┬───────────────────┘        └──────────────────────────────┘
           │ NO
           ▼
┌──────────────────────────────┐  YES   ┌──────────────────────────────┐
│  Finding missing data?       ├───────►│  NOT EXISTS / IS NULL        │
│                              │        │                              │
└──────────┬───────────────────┘        └──────────────────────────────┘
           │ NO
           ▼
┌──────────────────────────────┐  YES   ┌──────────────────────────────┐
│  Removing duplicates?        ├───────►│  ROW_NUMBER / DISTINCT       │
│                              │        │                              │
└──────────────────────────────┘        └──────────────────────────────┘
```

All patterns above are covered in the <a href="sql_single_table_query_strategies.html"><strong>Single-Table Strategies Guide</strong></a>.

> **Still stuck?** Check if you actually need a subquery — go to <a href="#subquery-decision"><strong>Step 4: Do I Need a Subquery?</strong></a>

<hr style="border: 3px solid black;">

<a id='multi-table-path'></a>

## Step 2B: Multi-Table Path — Pick the JOIN

You have two or more tables. First, decide how to connect them.

```
┌─────────────────────────────────────────┐
│   MULTIPLE TABLES — How do they relate? │
└────────────────────┬────────────────────┘
                     │
                     ▼
┌──────────────────────────────────┐  YES   ┌────────────────────────────────┐
│  Same structure / similar data?  │───────▶│  UNION ALL (or UNION)          │
│  "combine", "merge", "together"  │        │  Stack rows from both tables   │
└──────────────┬───────────────────┘        └────────────────────────────────┘
               │ NO
               ▼
┌──────────────────────────────────┐  YES   ┌────────────────────────────────┐
│  Table references itself?        │───────▶│  Self JOIN                     │
│  "manager", "parent", "reports"  │        │                                │
└──────────────┬───────────────────┘        └────────────────────────────────┘
               │ NO
               ▼
┌──────────────────────────────────┐  YES   ┌────────────────────────────────┐
│  Need ALL combinations?          │───────▶│  CROSS JOIN                    │
│  "every X with every Y"          │        │                                │
└──────────────┬───────────────────┘        └────────────────────────────────┘
               │ NO
               ▼
┌──────────────────────────────────┐  YES   ┌────────────────────────────────┐
│  Need to find what's MISSING?    │───────▶│  LEFT JOIN + WHERE IS NULL     │
│  "never", "didn't", "no match"   │        │  (or NOT EXISTS / ANTI JOIN)   │
└──────────────┬───────────────────┘        └────────────────────────────────┘
               │ NO
               ▼
┌──────────────────────────────────┐  YES   ┌────────────────────────────────┐
│  Must keep ALL rows from one     │───────▶│  LEFT JOIN                     │
│  table even if no match?         │        │  + COALESCE for NULL defaults  │
└──────────────┬───────────────────┘        └────────────────────────────────┘
               │ NO
               ▼
┌──────────────────────────────────┐        ┌────────────────────────────────┐
│  Default: Matching rows only     │───────▶│  INNER JOIN                    │
│  "get name", "look up", "find"   │        │  (most common starting point)  │
└──────────────────────────────────┘        └────────────────────────────────┘
```

All join patterns above are covered in the <a href="sql_multi_table_query_strategies.html"><strong>Multi-Table Strategies Guide</strong></a>.

> **After picking the JOIN:** Does the problem also need aggregation, ranking, or conditional logic? If yes, go to <a href="#combo-path"><strong>Step 2C</strong></a>. If the join alone answers the question, you're done.

<hr style="border: 3px solid black;">

<a id='combo-path'></a>

## Step 2C: After the JOIN — Pick the Transformation

You've picked your join from <a href='#multi-table-path'>Step 2B</a>. Now treat the joined result as a single table and pick the transformation.

```
┌─────────────────────────────────────────────────────────┐
│  You have joined data. What transformation is needed?   │
└───────────────────────┬─────────────────────────────────┘
                        │
    ┌───────────────────┼──────────────────┬──────────────────┐
    ▼                   ▼                  ▼                  ▼
┌────────────────┐ ┌────────────────┐ ┌────────────────┐ ┌────────────────┐
│  Rate / Ratio  │ │  Count / Sum   │ │  Rank across   │ │  Filter groups │
│  Conditional   │ │  per entity    │ │  joined data   │ │  by threshold  │
│  metric        │ │  (incl. zeros) │ │                │ │                │
│                │ │                │ │                │ │                │
│  GROUP BY      │ │  GROUP BY      │ │  ROW_NUMBER    │ │  GROUP BY      │
│  + CASE WHEN   │ │  + COUNT/SUM   │ │  RANK / LEAD   │ │  + HAVING      │
│                │ │                │ │  OVER()        │ │                │
└────────────────┘ └────────────────┘ └────────────────┘ └────────────────┘
```

All combo patterns above are covered in the <a href="sql_combined_strategy_patterns.html"><strong>Combined Strategy Patterns Guide</strong></a>.

> **Does your transformation also need a subquery?** Go to <a href="#subquery-decision"><strong>Step 4: Do I Need a Subquery?</strong></a>

<hr style="border: 3px solid black;">

<a id='date-fork'></a>

## Step 3: Date Fork

If your problem involves dates, this fork tells you which date tool to use. After choosing, **go back to the main tree** (<a href="#single-table-path">Step 2A</a> or <a href="#multi-table-path">Step 2B</a>) for the structural pattern.

```
┌──────────────────────────────────────────────────────────────────────┐
│                    What kind of date work?                           │
└──────┬───────────────────────┬───────────────────────┬───────────────┘
       ▼                       ▼                       ▼
┌──────────────┐        ┌──────────────┐        ┌──────────────┐
│  Comparing   │        │  Extracting  │        │  Calculating │
│  consecutive │        │  parts?      │        │  difference? │
│  rows?       │        │              │        │              │
│  (yesterday, │        │  (per month, │        │  (days       │
│   next day)  │        │   per year)  │        │   between,   │
│              │        │              │        │   gaps)      │
└──────┬───────┘        └──────┬───────┘        └──────┬───────┘
       ▼                       ▼                       ▼
┌──────────────┐        ┌──────────────┐        ┌──────────────┐
│  LAG / LEAD  │        │  EXTRACT /   │        │  DATEDIFF /  │
│  + date math │        │  DATE_PART   │        │  subtraction │
│              │        │  + GROUP BY  │        │  + LAG/LEAD  │
└──────────────┘        └──────────────┘        └──────────────┘
```

> **Key insight:** Date problems almost always combine with another pattern. The date fork tells you *which date tool* you need, then you still flow into the main tree for the structural pattern.

<hr style="border: 3px solid black;">

<a id='subquery-decision'></a>

## Step 4: Do I Need a Subquery?

After identifying your main pattern, check if a subquery is needed.

```
┌─────────────────────────────────────────────────────────┐
│  Does my calculation need a value from a DIFFERENT      │
│  aggregation level or a different table that I'm NOT    │
│  joining to?                                            │
│                                                         │
│  Examples:                                              │
│  • "percentage of total" (group count / global count)   │
│  • "above average" (row value vs. table average)        │
│  • "pre-aggregate then join" (summary + detail)         │
└──────────────────────┬──────────────────────────────────┘
                       │
          ┌────────────┴────────────┐
          │ NO                      │ YES
          ▼                         ▼
┌──────────────────┐   ┌───────────────────────────────────────┐
│  No subquery     │   │  Can I use a window function instead? │
│  needed — done!  │   │  (AVG() OVER(), SUM() OVER())         │
└──────────────────┘   └──────────────────┬────────────────────┘
                                          │
                             ┌────────────┴────────────┐
                             │ YES                     │ NO
                             ▼                         ▼
                  ┌──────────────────┐    ┌───────────────────────────┐
                  │  Window function │    │  YES — YOU NEED A         │
                  │  is cleaner —    │    │  SUBQUERY                 │
                  │  use that        │    │                           │
                  └──────────────────┘    │  Where does it go?        │
                                          └─────────────┬─────────────┘
                                                        │
                         ┌──────────────────────────────┼──────────────────────────────┐
                         ▼                              ▼                              ▼
          ┌──────────────────────────┐   ┌──────────────────────────┐   ┌──────────────────────────┐
          │  In the SELECT clause    │   │  In the WHERE clause     │   │  In the FROM clause      │
          │                          │   │                          │   │                          │
          │  "Show a global value    │   │  "Filter rows using a    │   │  "Pre-aggregate, then    │
          │   alongside each row"    │   │   computed threshold"    │   │   query the result"      │
          │                          │   │                          │   │                          │
          │  Signal words:           │   │  Signal words:           │   │  Signal words:           │
          │  percentage of total,    │   │  above average,          │   │  rank within groups,     │
          │  ratio, compared to all  │   │  greater than median,    │   │  filter pre-aggregated   │
          │                          │   │  top performers          │   │  results, join summaries │
          │                          │   │                          │   │                          │
          │  Pattern:                │   │  Pattern:                │   │  Pattern:                │
          │  COUNT(x) /              │   │  WHERE col >             │   │  SELECT * FROM           │
          │  (SELECT COUNT(*)        │   │  (SELECT AVG(col)        │   │  (SELECT ... GROUP BY)   │
          │   FROM ...) * 100        │   │   FROM ...)              │   │  AS subquery             │
          └──────────────────────────┘   └──────────────────────────┘   └──────────────────────────┘
```

For detailed examples of each subquery placement, see:
- <a href="sql_subqueries_select.html"><strong>Subqueries in SELECT</strong></a> — percentage of total, ratio calculations
- <a href="sql_subqueries_where.html"><strong>Subqueries in WHERE</strong></a> — filtering by computed thresholds
- <a href="sql_subqueries_from.html"><strong>Subqueries in FROM</strong></a> — pre-aggregating before further logic

<hr style="border: 3px solid black;">

<a id='quick-reference'></a>

## Quick Reference: Signal Words → Pattern

When you're stuck, scan the problem for these words:

| Signal Words in Problem | Pattern | Guide |
|---|---|---|
| "filter", "where", "only rows that" | WHERE + ORDER BY | <a href="sql_single_table_query_strategies.html">Single-Table</a> |
| "previous", "next", "consecutive", "change from" | LAG / LEAD | <a href="sql_single_table_query_strategies.html">Single-Table</a> |
| "per month", "per year", "extract" | DATE functions + GROUP BY | <a href="sql_single_table_query_strategies.html">Single-Table</a> |
| "total", "average", "count per", "sum of" | GROUP BY + aggregate | <a href="sql_single_table_query_strategies.html">Single-Table</a> |
| "top N", "rank", "nth highest" | ROW_NUMBER / RANK | <a href="sql_single_table_query_strategies.html">Single-Table</a> |
| "running total", "cumulative" | SUM() OVER() | <a href="sql_single_table_query_strategies.html">Single-Table</a> |
| "never", "didn't", "no match", "missing" | NOT EXISTS / LEFT JOIN IS NULL | <a href="sql_multi_table_query_strategies.html">Multi-Table</a> |
| "duplicates", "unique", "first occurrence" | ROW_NUMBER / DISTINCT | <a href="sql_single_table_query_strategies.html">Single-Table</a> |
| "label", "categorize", "if...then" | CASE WHEN | <a href="sql_single_table_query_strategies.html">Single-Table</a> |
| "groups having", "at least N" | GROUP BY + HAVING | <a href="sql_single_table_query_strategies.html">Single-Table</a> |
| "pivot", "rows to columns" | CASE + GROUP BY | <a href="sql_single_table_query_strategies.html">Single-Table</a> |
| "look up", "get name from", "enrich" | INNER JOIN | <a href="sql_multi_table_query_strategies.html">Multi-Table</a> |
| "all X even if no Y", "include unmatched" | LEFT JOIN | <a href="sql_multi_table_query_strategies.html">Multi-Table</a> |
| "every X with every Y", "all combinations" | CROSS JOIN | <a href="sql_multi_table_query_strategies.html">Multi-Table</a> |
| "combine", "merge tables", "stack" | UNION ALL | <a href="sql_multi_table_query_strategies.html">Multi-Table</a> |
| "manager", "parent", "reports to" | Self JOIN | <a href="sql_multi_table_query_strategies.html">Multi-Table</a> |
| "users who bought X", "exists in" | EXISTS / IN | <a href="sql_multi_table_query_strategies.html">Multi-Table</a> |
| "percentage of total", "ratio to all" | Scalar subquery in SELECT | <a href="sql_subqueries_select.html">Subquery SELECT</a> |
| "above average", "greater than median" | Scalar subquery in WHERE | <a href="sql_subqueries_where.html">Subquery WHERE</a> |
| "rate per entity including zeros" | LEFT JOIN + GROUP BY + CASE | <a href="sql_combined_strategy_patterns.html">Combined</a> |
| "count per entity including zeros" | LEFT JOIN + GROUP BY + COUNT | <a href="sql_combined_strategy_patterns.html">Combined</a> |

<hr style="border: 3px solid black;">

<a id='full-flow'></a>

## The Complete Flow — All Steps on One Diagram

```
┌─────────────────────────────────────────────────────────┐
│                  READ THE PROBLEM                       │
└────────────────────────┬────────────────────────────────┘
                         │
                  How many tables?
                         │
            ┌────────────┴────────────────┐
            ▼                             ▼
     ┌────────────┐              ┌──────────────────┐
     │  ONE TABLE │              │  2+ TABLES       │
     └─────┬──────┘              └────────┬─────────┘
           │                              │
           ▼                              ▼
  ┌─────────────────┐          ┌───────────────────────┐
  │  Walk Step 2A   │          │  Walk Step 2B         │
  │  (single-table  │          │  (pick the JOIN)      │
  │   patterns)     │          └───────────┬───────────┘
  └────────┬────────┘                      │
           │                     Need transformation
           │                     after the join?
           │                               │
           │                    ┌──────────┴──────────┐
           │                    │ NO                  │ YES
           │                    ▼                     ▼
           │              ┌──────────┐     ┌───────────────────┐
           │              │  Done!   │     │  Walk Step 2C     │
           │              │  JOIN    │     │  (pick transform) │
           │              │  alone   │     └─────────┬─────────┘
           │              │  answers │               │
           │              │  it      │               │
           │              └──────────┘               │
           │                                         │
           └─────────────────┬───────────────────────┘
                             │
                             ▼
              ┌───────────────────────────────┐
              │  Does any part of my formula  │
              │  need a value from a          │
              │  DIFFERENT aggregation level? │
              └──────────────┬────────────────┘
                             │
                ┌────────────┴────────────┐
                │ NO                      │ YES
                ▼                         ▼
        ┌──────────────┐       ┌───────────────────┐
        │  DONE!       │       │  Walk Step 4      │
        │  Write your  │       │  (subquery        │
        │  query       │       │   placement)      │
        └──────────────┘       └───────────────────┘
```

<hr style="border: 3px solid black;">

<a id='worked-example'></a>

## Worked Example: Walking the Tree

**Problem:** Find the percentage of total users registered in each contest, rounded to two decimal places. Order by percentage descending, then contest_id ascending.

**Tables:** Users (user_id, user_name) and Register (contest_id, user_id)

---

### Walking the tree:

**Step 1 — How many tables?** Two (Users and Register) → Go to Step 2B

**Step 2B — Pick the JOIN:**
- Same structure? No
- Self-referencing? No
- All combinations? No
- Finding missing? No
- Keep all rows? No
- Matching rows only? Hmm... do I actually need columns from Users?

**Pause:** The output only needs `contest_id` and `percentage`. I don't need `user_name` or any other column from Users. I just need a **count** from it. → No JOIN needed, but I do need data from another table.

**Step 2C — Transformation:** GROUP BY `contest_id` + COUNT per group → GROUP BY + aggregate

**Step 4 — Do I need a subquery?** My formula is `group_count / total_count * 100`. The `total_count` comes from a different table at a different grain. Can I use a window function? No — it's a different table entirely. → **Subquery needed, in the SELECT clause** (denominator role).

---

### Final query:

```sql
SELECT r.contest_id,
       ROUND(
           (COUNT(DISTINCT r.user_id)::numeric
            / (SELECT COUNT(DISTINCT user_id) FROM Users)
           ) * 100, 2
       ) AS percentage
FROM Register AS r
GROUP BY r.contest_id
ORDER BY percentage DESC, r.contest_id ASC;
```

**Tree path:** Step 1 (2 tables) → Step 2B (no join needed) → Step 2C (GROUP BY) → Step 4 (scalar subquery in SELECT)

<hr style="border: 3px solid black;">

<a id='final-takeaway'></a>

## Final Takeaway

Every SQL problem breaks down into the same four questions:

1. **How many tables?** → Determines if you need a join
2. **What join?** → Match the relationship pattern to the right join type
3. **What transformation?** → Filter, aggregate, rank, compare, or label
4. **Different grain?** → If yes, you need a subquery (and now you know where it goes)

Practice walking the tree on every problem before writing code. After a few dozen problems, the routing becomes automatic.